<a href="https://colab.research.google.com/github/Mohrezasharifi/10DaysOfCodeVvce/blob/master/FineGrainedClassificationFinall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision gradio pandas matplotlib

# Import libraries
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image
import gradio as gr
import matplotlib.pyplot as plt
import time
from typing import Tuple

In [2]:
# Mount Google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Set dataset path (update this to your dataset location)
root_dir = "/content/drive/MyDrive/CUB_200_2011/CUB_200_2011/CUB_200_2011"

# Verify dataset structure
print("Dataset contents:")
!ls -la "{root_dir}"

# Check required files
required_files = [
    'images.txt',
    'image_class_labels.txt',
    'train_test_split.txt',
    'bounding_boxes.txt',
    'classes.txt'
]

for file in required_files:
    path = os.path.join(root_dir, file)
    if not os.path.exists(path):
        print(f"Missing file: {path}")
    else:
        print(f"Found: {path}")

# Verify images folder
images_path = os.path.join(root_dir, 'images')
if os.path.exists(images_path):
    print(f"Found {len(os.listdir(images_path))} subdirectories in images folder")
else:
    print("Missing images folder")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset contents:
total 1199
drwx------   2 root root   4096 Feb 12 11:22 attributes
-rw-------   1 root root 328018 Jul 25  2011 bounding_boxes.txt
-rw-------   1 root root   4824 Jul 25  2011 classes.txt
-rw-------   1 root root   8196 Feb 11 23:18 .DS_Store
-rw-------   1 root root 100487 Jul 25  2011 image_class_labels.txt
drwx------ 202 root root   4096 Feb 12 11:22 images
-rw-------   1 root root 682287 Jul 25  2011 images.txt
drwx------   2 root root   4096 Feb 12 11:22 parts
-rw-------   1 root root   6282 Jul 29  2011 README
-rw-------   1 root root  83198 Jul 25  2011 train_test_split.txt
Found: /content/drive/MyDrive/CUB_200_2011/CUB_200_2011/CUB_200_2011/images.txt
Found: /content/drive/MyDrive/CUB_200_2011/CUB_200_2011/CUB_200_2011/image_class_labels.txt
Found: /content/drive/MyDrive/CUB_200_2011/CUB_200_2011/CUB_200_2011/train_test_split.txt
Fou

In [3]:
class CUBDataset(Dataset):
    def __init__(self, root_dir, transform=None, train=True):
        self.root_dir = root_dir
        self.transform = transform

        # Load metadata files
        self.images = pd.read_csv(os.path.join(root_dir, 'images.txt'),
                                sep=' ', names=['img_id', 'img_path'])
        self.labels = pd.read_csv(os.path.join(root_dir, 'image_class_labels.txt'),
                                sep=' ', names=['img_id', 'class_id'])
        self.train_test = pd.read_csv(os.path.join(root_dir, 'train_test_split.txt'),
                                    sep=' ', names=['img_id', 'is_training'])
        self.bboxes = pd.read_csv(os.path.join(root_dir, 'bounding_boxes.txt'),
                                sep=' ', names=['img_id', 'x', 'y', 'width', 'height'])

        # Merge data
        self.data = self.images.merge(self.labels, on='img_id')
        self.data = self.data.merge(self.train_test, on='img_id')
        self.data = self.data.merge(self.bboxes, on='img_id')

        # Filter and prepare data
        self.data = self.data[self.data['is_training'] == (1 if train else 0)]
        self.data['class_id'] = self.data['class_id'] - 1  # 0-based index
        self.class_names = pd.read_csv(os.path.join(root_dir, 'classes.txt'),
                                    sep=' ', names=['class_id', 'class_name'])['class_name'].tolist()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.root_dir, 'images', row['img_path'])

        # Load and crop image
        img = Image.open(img_path).convert('RGB')
        bbox = (row['x'], row['y'], row['x']+row['width'], row['y']+row['height'])
        img = img.crop(bbox)

        if self.transform:
            img = self.transform(img)

        return img, row['class_id']

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(448),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(448),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

In [5]:
def prepare_data():
    full_train = CUBDataset(root_dir, transform=data_transforms['train'], train=True)

    # Split into train/val
    train_size = int(0.9 * len(full_train))
    val_size = len(full_train) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_train, [train_size, val_size],
        generator=torch.Generator().manual_seed(42))

    test_dataset = CUBDataset(root_dir, transform=data_transforms['test'], train=False)

    return {
        'train': DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2),
        'val': DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2),
        'test': DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)
    }, full_train.class_names

# Initialize data loaders
dataloaders, class_names = prepare_data()

In [6]:
def create_model():
    model = models.efficientnet_b4(pretrained=True)

    # Freeze early layers
    for param in model.features[:-6].parameters():
        param.requires_grad = False

    # Modify classifier
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 200))
    return model.to(device)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_model()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B4_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B4_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
import torch.optim as optim
from torch.optim import lr_scheduler
import copy
def train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs=25):
    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs-1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        scheduler.step()

    model.load_state_dict(best_model_wts)
    return model

# Define loss, optimizer, and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# Train the model
model = train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs=10)

Epoch 0/9
----------


In [6]:
# Save model
import torch
from torchvision import transforms, models
torch.save(model.state_dict(), "/content/drive/MyDrive/bird_model.pth")

NameError: name 'model' is not defined

In [1]:
@torch.inference_mode()
def classify_bird(image: Image.Image) -> Tuple[dict, Image.Image]:
    start_time = time.time()

    try:
        # Preprocess
        transform = data_transforms['test']
        input_tensor = transform(image).unsqueeze(0).to(device)

        # Predict
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

        # Get top predictions
        top_probs, top_ids = torch.topk(probabilities, 5)
        predictions = {class_names[i]: float(p) for p, i in zip(top_probs, top_ids)}

        # Create visualization
        plt.figure(figsize=(8, 6))
        plt.barh(range(5), list(predictions.values())[::-1])
        plt.yticks(range(5), list(predictions.keys())[::-1])
        plt.title("Top 5 Predictions")
        plt.tight_layout()
        plt.savefig("/tmp/prediction_viz.png")
        plt.close()

        print(f"Prediction time: {time.time()-start_time:.2f}s")
        return predictions, "/tmp/prediction_viz.png"

    except Exception as e:
        print(f"Error during prediction: {e}")
        return {"Error": 1.0}, None

# Create Gradio interface
interface = gr.Interface(
    fn=classify_bird,
    inputs=gr.Image(type="pil", label="Upload Bird Image"),
    outputs=[
        gr.Label(num_top_classes=5, label="Predictions"),
        gr.Image(label="Visualization", type="filepath")
    ],
    examples=[
        os.path.join(root_dir, "images/001.Black_footed_Albatross/Black_Footed_Albatross_0001_796111.jpg"),
        os.path.join(root_dir, "images/002.Laysan_Albatross/Laysan_Albatross_0001_545.jpg")
    ],
    title="Bird Species Classification",
    description="Fine-grained classification of 200 bird species",
    allow_flagging="never"
)

# Launch interface
interface.launch(share=True)

NameError: name 'torch' is not defined

In [ ]:


# Load model (for future use)
model.load_state_dict(torch.load("/content/drive/MyDrive/bird_model.pth"))
model.eval()